# 🎙️ TurboVoiceCloner: Professional TTS Server
**Status:** Error Fixed | **GPU:** T4 Required

### 🛠️ महत्वपूर्ण निर्देश:
1. Runtime → Change runtime type → **T4 GPU** चुनें।
2. Step 1 चलाने के बाद, Step 2 चलाएं और लाल पट्टी (Red Bar) हटने का इंतज़ार करें।

In [ ]:
# @title 🚀 Step 1: Install & Repair Engine
# @markdown यह सेल उन फाइलों को ठीक करेगा जो 'ImportError' दे रही थीं।
import os
os.environ['MPLBACKEND'] = 'Agg'

print("Installing Core Audio Tools...")
!git clone https://github.com/devnen/Chatterbox-TTS-Server.git 2>/dev/null || echo "Repo exists"
%cd /content/Chatterbox-TTS-Server

# Stable CUDA 12.1 Installation
!pip install torch==2.5.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install git+https://github.com/devnen/chatterbox.git -q
!pip install fastapi uvicorn gradio==4.44.1 pydub python-multipart requests -q

# @markdown **Fixing the Model Selection Error**
engine_path = '/content/Chatterbox-TTS-Server/engine.py'
if os.path.exists(engine_path):
    with open(engine_path, 'r') as f: content = f.read()
    # 'chatterbox-turbo' को 'chatterbox-original' से बदलें ताकि ImportError न आए
    content = content.replace("'chatterbox-turbo'", "'chatterbox-original'")
    with open(engine_path, 'w') as f: f.write(content)

print("✅ Engine Patched & Components Installed!")

In [ ]:
# @title ⚙️ Step 2: Start Stable TTS Server
# @markdown सर्वर शुरू होने के बाद 40 सेकंड प्रतीक्षा करें।
import threading, time, uvicorn, gc
from google.colab.output import serve_kernel_port_as_window

!fuser -k 8004/tcp 2>/dev/null || echo "Port clear"
gc.collect()

def run_server():
    from server import app
    # Safe mode launch
    uvicorn.run(app, host="0.0.0.0", port=8004, log_level="error")

print("🚀 Starting Turbo TTS Server... Model loading in GPU...")
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(40) # Wait for complete GPU loading
print("🎉 Server Ready! Click the window below:")
serve_kernel_port_as_window(8004)